# 9. Describing a dataset: summaries and Claude: exercises

Exercises of [chapter 9](../../09_describing_a_dataset.md). Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, difflib, json, shutil, tempfile
import rasterio
import yaml
from data_acquisition.catalog import build, describe, evidence
from data_acquisition.catalog import summary as summaries

OUT = ROOT / "output/manual/describe/training_pond"
REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())


def check(s):
    """Check a summary (a dict) and build it in memory; print what the pipeline says."""
    path = Path(tempfile.mkdtemp()) / "my_summary.yaml"
    path.write_text(yaml.safe_dump(copy.deepcopy(s), sort_keys=False))
    try:
        loaded = summaries.load(str(path))
    except ValueError as e:
        print("BLOCKS:\n", str(e).split("\n", 1)[1])
        return None
    catalog, report = build.build(SAMPLE, loaded)
    print("valid; build report:", {k: v for k, v in report.items() if v} or "empty")
    return catalog

**1. Evidence.** Gather the evidence and the prompt for the sample dataset into `output/manual/describe/training_pond`. For each file, what did code read? What did it read about the photos, and why so little?

> **From the chapter.** §2: code gathers what it can read for sure, file by file.
>
> **Code.** `describe prompt`, a command of [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py), run with `subprocess`; it gathers with [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py) and writes `evidence.json` and `prompt.md` into `OUT`.

In [ ]:
...  # your code here: describe prompt --root SAMPLE --folders "repo/*" --dataset training_pond

**2. The rules.** Read the rules in `prompt.md`. Which of them stop Claude from inventing values?

> **From the chapter.** §3: the drafting rules that matter most.
>
> **Code.** `prompt.md` in `OUT`, or `describe.RULES` in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py).

In [ ]:
...  # your code here (describe.RULES)

**3. By hand.** Write a summary for the sample from the evidence alone, starting from the skeleton in the notebook; make it pass `check`; compare it with the reference summary.

> **From the chapter.** §1: the parts of a summary. [Chapter 6](../../06_what_the_pipeline_needs.md) lists every field, and what happens without it.
>
> **Code.** `check(s)`, defined above, calls `summaries.load` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py) and `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py). Compare with [`docs/manual/solutions/training_pond.yaml`](../training_pond.yaml).

In [ ]:
mine = {
    "schema": 1, "dataset": "training_pond", "title": "...",
    "authored": {"by": "A Student", "date": datetime.date.today()}, "reviewed": None,
    "source": {"kind": "dataone", "provider": "..."},
    "groups": [{"folder": "repo/doi_10.5555_training.pond", "record": {"path": "metadata.xml", "format": "eml"},
                "site": "training_pond", "files": [
                    # one entry per file: {"path": ..., "product": ..., "date": datetime.date(...)} or role + attach_to
                ]}],
}
check(mine)

**4. Review a draft.** [`docs/manual/data/flawed_draft.yaml`](../../data/flawed_draft.yaml) passes the check but has four mistakes a reviewer should catch. Find them with the evidence.

> **From the chapter.** §4: a draft that passes the check is valid, not yet right. The review compares it with the evidence.
>
> **Code.** [`docs/manual/data/flawed_draft.yaml`](../../data/flawed_draft.yaml), `check(s)` defined above, and the evidence of exercise 1.

In [ ]:
flawed = yaml.safe_load((ROOT / "docs/manual/data/flawed_draft.yaml").read_text())
check(flawed)
...  # your code here: compare its values with the evidence (headers, record, methods text)

**5. Load and check.** Load the reference summary with `summaries.load`: what is added to it? Then run `summaries.check` on the file as written and on the loaded summary, and on copies with a missing title, an unknown product and an unknown field. What does `check` expect?

> **From the chapter.** §5, "`load` changes the summary": `check` must be given the summary as written.
>
> **Code.** `summaries.load` and `summaries.check`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
path = ROOT / "docs/manual/solutions/training_pond.yaml"
...  # your code here (summaries.load, yaml.safe_load, summaries.check)

**6. The registry.** Read the sample's registry and the real one ([`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml)) with `summaries.load_sites`. How many sites does each have? Extract the `name` of every site of the real one as a list and as a dict by slug, and list its regions. Then break a copy of the sample's registry (a CRS without `EPSG:`, a `land_status` that is not null or an entry, a site `contains` one that does not exist) and run `check_sites`. Finally, which sites does the reference summary use (`used_sites`)?

> **From the chapter.** §5: the registry functions. [Chapter 5](../../05_data_governance.md) explains the manual fields.
>
> **Code.** `summaries.load_sites`, `summaries.check_sites` and `summaries.used_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
sample = summaries.load_sites()
real = summaries.load_sites(ROOT / "data_acquisition/sites.yaml")
...  # your code here

**7. Anchors.** YAML lets you write a value once (`&name`) and reuse it (`*name`). Load a short text that does, and see what `yaml.safe_load` returns. Then run `summaries.anchors` on it, and on a text with `5 * 3 & more` in a note. Why must `anchors` look at the text, not the loaded dictionary?

> **From the chapter.** §5: a summary must write every value where it applies, so YAML anchors are refused.
>
> **Code.** `summaries.anchors`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `yaml.safe_load` is the YAML library's.

In [ ]:
with_anchor = "base: &common\n  platform: UAV\nfile_a: *common\n"
...  # your code here

**8. A file's time.** For every data file of the reference summary: `resolve` it with its group, compute `summaries.epoch`, and print the path, start, end and label. Then compute `epoch` for a day, a month, a year, a range and an unknown date, and make it fail five ways.

> **From the chapter.** §5, on `epoch`: a file's time is a date with its precision, or a start and an end.
>
> **Code.** `summaries.resolve` and `summaries.epoch`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
group = s["groups"][0]
...  # your code here

**9. A layer's time.** A vector file can hold one layer per date, named by it. With `summaries.layer_epoch` and the pattern `_(\d{4})$` (year), find the date of the layers `Dams_2010`, `Dams_2013_old`, `notes` and `Dams_0000`. Then date `D26Jul_2018-07-26` by day and `x_2023-07` by month, and try `x_2023-02-30`. When does it answer None?

> **From the chapter.** §5, and [chapter 6](../../06_what_the_pipeline_needs.md), §3: a vector file can hold one date per layer, named in the layer.
>
> **Code.** `summaries.layer_epoch`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
spec = {"pattern": r"_(\d{4})$", "date_precision": "year"}
...  # your code here

**10. Who wrote it.** Call `summaries.author` on the reference summary. Then on copies with `reviewed` set, and with a person as author. What does `summaries.by_claude` say about `claude`, `claude (claude-opus-5-5)`, `Claudia Rossi` and `Federico`?

> **From the chapter.** §4: a person sets `reviewed`, and the catalogue's labels follow it.
>
> **Code.** `summaries.author` and `summaries.by_claude`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
...  # your code here

**11. Evidence in code.** In a scratch folder with a copy of the orthomosaic and a sidecar, what does `evidence.entries` list? On the sample's files, plus a broken `.tif` and an unknown `.bin`, what does `evidence.describe_file` return? Then run `evidence.gather` on the sample, and on a folder of 10 photos: why is that one entry, when the sample's 6 photos are six?

> **From the chapter.** §2: what evidence is gathered, and §5: its functions.
>
> **Code.** `evidence.entries`, `evidence.describe_file`, `evidence.gather` and `evidence.PHOTO_FOLDER`, in [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
...  # your code here (evidence.entries, evidence.describe_file, evidence.gather, evidence.PHOTO_FOLDER)

**12. The prompt and the check in code.** Print `describe.registry_lines()`. Write a prompt into a scratch folder with `describe.write_prompt`: which files appear, and what did it fill in? Then run `describe.check_summary` on the reference summary, on a copy that names a missing file and leaves the DSM out, and on one with an unknown product. What does it return each time?

> **From the chapter.** §3: the prompt and the check, and §5: their functions.
>
> **Code.** `describe.registry_lines`, `describe.write_prompt` and `describe.check_summary`, in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py); `evidence.gather` in [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py).

In [ ]:
found = evidence.gather(SAMPLE, ["repo/*"])
...  # your code here

**13. (Optional) Draft with Claude.** Run the prompt in a Claude Code session on the sample. The prompt writes into [`data_acquisition/summaries/`](../../../../data_acquisition/summaries) and may add a site to [`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml): delete the draft and undo any registry change afterwards (`git status`, `git restore`).

> **From the chapter.** §3: drafting in a new Claude Code session.
>
> **Code.** `prompt.md` from exercise 1, in a Claude Code session. It may write into [`data_acquisition/summaries/`](../../../../data_acquisition/summaries) and [`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml).

In [ ]:
# Done in a Claude Code session, not here. Afterwards:
print(subprocess.run(["git", "status", "--short"], cwd=ROOT, capture_output=True, text=True).stdout)

**14. Find it in the code.** Where are the drafting rules, and which function decides what evidence a raster gets?

> **From the chapter.** §3 for the rules, §2 for the evidence.
>
> **Code.** Look in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py) and [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py) (imported above as `describe` and `evidence`).

In [ ]:
import inspect
...  # your code here